# BirdCLEF+ 2026 - Training Pipeline

**Hardware:** Kaggle dual NVIDIA Tesla T4 (2 x 16 GB VRAM)
**Prerequisite:** Run `01_preprocessing` first and attach its output as an external dataset.

### Key Differentiators
1. **Multi-scale spectrogram frontend** - dual n_fft for time vs frequency resolution
2. **Focal loss** - handles rare species better than plain BCE
3. **Sqrt-frequency class-balanced sampling** - rare species get more training
4. **Model Soup** - ensemble-quality at single-model inference cost
5. **3 diverse backbones x 3 folds** - true architecture diversity
6. **GeM + SED attention head** - attends to bird call frames

In [1]:
!pip install -q timm torchaudio librosa tqdm psutil

## 0 - Imports & Configuration

In [2]:
import ast
import gc
import json
import math
import os
import random
import time
import warnings
from copy import deepcopy
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import librosa
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

import torch
import torch.distributed as dist
import torch.multiprocessing as mp
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torch.utils.data.distributed import DistributedSampler

import timm

warnings.filterwarnings('ignore')

In [3]:
CFG: Dict = dict(
    # -- Paths ----------------------------------------------------------------
    # IMPORTANT: Update this path after attaching 01_preprocessing output
    preprocessed_dir = Path('/kaggle/input/datasets/shishiradhikari11/birdclef-2026-preprocessed'),
    output_dir       = Path('/kaggle/working'),

    # -- Audio frontend -------------------------------------------------------
    sample_rate    = 32_000,
    clip_duration  = 5,
    n_fft_lo       = 2048,    # High frequency resolution (tonal whistles)
    hop_length_lo  = 512,
    n_fft_hi       = 512,     # High time resolution (rapid trills/clicks)
    hop_length_hi  = 128,
    n_mels         = 128,
    fmin           = 40,
    fmax           = 14_000,

    # -- Classification -------------------------------------------------------
    num_classes = None,  # set from label_map.json

    # -- Training schedule ----------------------------------------------------
    batch_size       = 64,
    num_workers      = 2,
    epochs           = 30,
    n_folds          = 5,
    train_folds      = [0, 1, 2],  # 3 folds for diversity
    lr               = 1e-3,
    weight_decay     = 1e-4,
    backbone_lr_mult = 0.1,

    # -- 3 diverse backbones --------------------------------------------------
    backbones = [
        'tf_efficientnet_b0_ns',   # Fast, proven
        'eca_nfnet_l0',            # Different family, strong on audio
        'tf_efficientnetv2_b3',    # Larger, captures more complex patterns
    ],

    # -- Augmentation ---------------------------------------------------------
    mixup_prob       = 0.5,
    mixup_alpha      = 0.4,
    time_shift_prob  = 0.5,
    gain_prob        = 0.4,
    gain_db_min      = -12.0,
    gain_db_max      = 2.0,
    pink_noise_prob  = 0.3,
    pink_noise_snr_min = 15.0,
    pink_noise_snr_max = 35.0,
    freq_mask_param  = 16,
    time_mask_param  = 32,
    n_freq_masks     = 2,
    n_time_masks     = 2,

    # -- Focal loss -----------------------------------------------------------
    focal_gamma = 2.0,
    focal_alpha = 0.25,

    # -- Model Soup -----------------------------------------------------------
    soup_epochs = [14, 16, 18, 20],
)

SAMPLES_PER_CLIP = CFG['sample_rate'] * CFG['clip_duration']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS = torch.cuda.device_count()
CFG['output_dir'].mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


seed_everything(42)
print(f'[ENV] Device: {DEVICE} | GPUs: {N_GPUS}')

[ENV] Device: cpu | GPUs: 0


## 1 - Multi-Scale Mel Frontend

Different bird vocalizations need different time-frequency tradeoffs:
- **Tonal whistles/songs** -> high frequency resolution (large `n_fft=2048`)
- **Rapid trills/clicks** -> high time resolution (small `n_fft=512`)

We stack both as 2 channels -> `(B, 2, 128, T)` feeding into `in_chans=2` backbone.
Simpler than learnable PCEN, more robust, zero learnable parameters in the frontend.

In [4]:
class MultiScaleMelFrontend(nn.Module):
    """Dual-resolution spectrogram frontend with InstanceNorm."""

    def __init__(self, cfg: Dict) -> None:
        super().__init__()
        sr = cfg['sample_rate']
        n_mels = cfg['n_mels']
        fmin, fmax = cfg['fmin'], cfg['fmax']

        self.n_fft_lo = cfg['n_fft_lo']
        self.hop_lo = cfg['hop_length_lo']
        self.n_fft_hi = cfg['n_fft_hi']
        self.hop_hi = cfg['hop_length_hi']
        self.n_mels = n_mels
        self.eps = 1e-6

        mel_lo = librosa.filters.mel(
            sr=sr, n_fft=self.n_fft_lo, n_mels=n_mels,
            fmin=fmin, fmax=fmax, norm='slaney', htk=False,
        ).astype(np.float32)
        mel_hi = librosa.filters.mel(
            sr=sr, n_fft=self.n_fft_hi, n_mels=n_mels,
            fmin=fmin, fmax=fmax, norm='slaney', htk=False,
        ).astype(np.float32)
        self.register_buffer('mel_fb_lo', torch.from_numpy(mel_lo))
        self.register_buffer('mel_fb_hi', torch.from_numpy(mel_hi))

        self.register_buffer('win_lo', torch.hann_window(self.n_fft_lo))
        self.register_buffer('win_hi', torch.hann_window(self.n_fft_hi))

        self.norm = nn.InstanceNorm2d(2, affine=False)

    def _mel_spec(self, wav, n_fft, hop, win, mel_fb):
        stft = torch.stft(
            wav, n_fft=n_fft, hop_length=hop,
            win_length=n_fft, window=win,
            return_complex=True, normalized=False,
        )
        power = stft.abs().pow(2)
        mel = torch.matmul(mel_fb, power)
        return torch.log(mel.clamp(min=self.eps))

    @torch.amp.autocast('cuda', enabled=False)
    def forward(self, wav: torch.Tensor) -> torch.Tensor:
        wav = wav.float()

        mel_lo = self._mel_spec(wav, self.n_fft_lo, self.hop_lo,
                                self.win_lo, self.mel_fb_lo)
        mel_hi = self._mel_spec(wav, self.n_fft_hi, self.hop_hi,
                                self.win_hi, self.mel_fb_hi)

        # Align time dimensions via interpolation
        T_lo, T_hi = mel_lo.shape[-1], mel_hi.shape[-1]
        T_target = max(T_lo, T_hi)

        if T_lo < T_target:
            mel_lo = F.interpolate(
                mel_lo.unsqueeze(1), size=(self.n_mels, T_target),
                mode='bilinear', align_corners=False,
            ).squeeze(1)
        if T_hi < T_target:
            mel_hi = F.interpolate(
                mel_hi.unsqueeze(1), size=(self.n_mels, T_target),
                mode='bilinear', align_corners=False,
            ).squeeze(1)

        x = torch.stack([mel_lo, mel_hi], dim=1)  # (B, 2, n_mels, T)
        x = self.norm(x)
        return x

## 2 - Dataset & Class-Balanced Sampling

In [5]:
class BirdDataset(Dataset):
    """Memory-mapped dataset with class-balanced sampling support."""

    def __init__(self, meta_df, npy_path, cfg, is_train=True):
        self.is_train = is_train
        self.npy_path = npy_path
        self.mmap_indices = meta_df['mmap_index'].values.astype(np.int64)
        self.label_ids = meta_df['label_id'].values.astype(np.int64)
        self._waveforms = None

        n = len(self.mmap_indices)
        num_classes = cfg['num_classes']

        # Primary targets (one-hot)
        self.primary_targets = torch.zeros(n, num_classes, dtype=torch.float32)
        self.primary_targets[torch.arange(n), torch.from_numpy(self.label_ids)] = 1.0

        # Secondary targets
        self.secondary_targets = torch.zeros(n, num_classes, dtype=torch.float32)
        lbl2id = cfg.get('label_map', {})
        if 'secondary_labels' in meta_df.columns:
            for i, sec_str in enumerate(meta_df['secondary_labels'].fillna('[]')):
                try:
                    sec_list = ast.literal_eval(sec_str)
                    for species in sec_list:
                        key = int(species) if isinstance(species, (int, float)) else species
                        if key in lbl2id:
                            self.secondary_targets[i, lbl2id[key]] = 1.0
                except Exception:
                    pass

    def _init_arrays(self):
        if self._waveforms is None:
            self._waveforms = np.load(self.npy_path, mmap_mode='r')

    def __len__(self):
        return len(self.mmap_indices)

    def __getitem__(self, index):
        self._init_arrays()
        mmap_idx = int(self.mmap_indices[index])
        wav = torch.from_numpy(self._waveforms[mmap_idx].copy())
        return wav, self.primary_targets[index], self.secondary_targets[index]


def build_class_weights(df, num_classes):
    """Sqrt-frequency class weights for balanced sampling."""
    counts = np.zeros(num_classes, dtype=np.float64)
    for lid in df['label_id'].values:
        counts[lid] += 1
    freq = counts / counts.sum()
    weights = 1.0 / np.sqrt(freq + 1e-8)
    weights /= weights.sum()
    sample_weights = torch.zeros(len(df), dtype=torch.float64)
    for i, lid in enumerate(df['label_id'].values):
        sample_weights[i] = weights[lid]
    return sample_weights

## 3 - Model Architecture

- **SEDHead**: GeM frequency pooling + temporal attention
- **BirdModel**: Frontend -> SpecAugment -> Backbone -> SEDHead

In [6]:
class SEDHead(nn.Module):
    """SED head with GeM frequency pooling + temporal attention.
    Dropout is inside the head (applied before classification)."""

    def __init__(self, in_features: int, num_classes: int):
        super().__init__()
        self.fc = nn.Linear(in_features, num_classes)
        self.attention = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.Tanh(),
            nn.Linear(128, num_classes),
        )
        self.dropout = nn.Dropout(0.3)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, F, T) from backbone
        # GeM pooling on frequency axis
        x = x.clamp(min=1e-6).pow(3.0).mean(dim=2).pow(1.0 / 3.0)  # (B, C, T)
        x = x.transpose(1, 2)  # (B, T, C)
        x = self.dropout(x)    # Dropout BEFORE classification
        framewise = self.fc(x)
        att_weights = torch.softmax(self.attention(x), dim=1)
        return torch.sum(framewise * att_weights, dim=1)


class BirdModel(nn.Module):
    """End-to-end bird audio classifier."""

    def __init__(self, cfg: Dict, backbone_name: str, is_train: bool = True):
        super().__init__()
        self.is_train = is_train

        self.frontend = MultiScaleMelFrontend(cfg)
        self.spec_aug = self._build_spec_augment(cfg)

        self.backbone = timm.create_model(
            backbone_name, pretrained=True,
            in_chans=2, num_classes=0, global_pool='',
        )

        self.head = SEDHead(
            in_features=self.backbone.num_features,
            num_classes=cfg['num_classes'],
        )

    def _build_spec_augment(self, cfg):
        import torchaudio.transforms as T
        layers = []
        for _ in range(cfg['n_freq_masks']):
            layers.append(T.FrequencyMasking(freq_mask_param=cfg['freq_mask_param']))
        for _ in range(cfg['n_time_masks']):
            layers.append(T.TimeMasking(time_mask_param=cfg['time_mask_param']))
        return nn.Sequential(*layers)

    def forward(self, wav: torch.Tensor) -> torch.Tensor:
        spec = self.frontend(wav)  # (B, 2, n_mels, T)

        if self.is_train and self.training:
            # Apply SAME mask to both channels (consistent masking)
            B, C, M, T = spec.shape
            ch0 = spec[:, 0:1, :, :]  # (B, 1, M, T)
            ch0_masked = self.spec_aug(ch0)
            mask = (ch0_masked != ch0).float()
            spec = spec * (1.0 - mask.expand_as(spec))

        features = self.backbone(spec)
        return self.head(features)

## 4 - Focal Loss with Secondary Label Masking

In [7]:
class FocalBCELoss(nn.Module):
    """Focal loss with secondary label masking.
    
    Focal loss down-weights easy negatives, crucial for imbalanced classification.
    When gamma=0, this reduces to standard BCE.
    """

    def __init__(self, gamma: float = 2.0, alpha: float = 0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, primary_targets, secondary_targets):
        combined = torch.clamp(primary_targets + secondary_targets, 0.0, 1.0)
        bce = F.binary_cross_entropy_with_logits(logits, combined, reduction='none')
        probs = torch.sigmoid(logits)
        p_t = probs * combined + (1 - probs) * (1 - combined)
        focal_weight = (1 - p_t).pow(self.gamma)
        alpha_t = self.alpha * combined + (1 - self.alpha) * (1 - combined)
        focal_loss = alpha_t * focal_weight * bce

        # Mask secondary labels
        mask = torch.ones_like(focal_loss)
        mask[secondary_targets == 1.0] = 0.0
        mask[primary_targets == 1.0] = 1.0

        return (focal_loss * mask).mean()

## 5 - GPU Waveform Augmentations

In [8]:
def batch_augment(wavs, targets, secondary, cfg):
    """Vectorized GPU waveform augmentations."""
    B, T = wavs.shape
    device = wavs.device

    # 1. Time shift
    if random.random() < cfg['time_shift_prob']:
        shifts = torch.randint(0, T, (B, 1), device=device)
        idx = (torch.arange(T, device=device).unsqueeze(0) - shifts) % T
        wavs = torch.gather(wavs, 1, idx)

    # 2. Random gain
    gain_mask = (torch.rand(B, 1, device=device) < cfg['gain_prob']).float()
    gain_db = torch.empty(B, 1, device=device).uniform_(
        cfg['gain_db_min'], cfg['gain_db_max']
    )
    linear_gain = 1.0 + (10.0 ** (gain_db / 20.0) - 1.0) * gain_mask
    wavs = torch.clamp(wavs * linear_gain, -1.0, 1.0)

    # 3. Pink noise (more realistic than white noise for nature recordings)
    if random.random() < cfg['pink_noise_prob']:
        freqs = torch.fft.rfftfreq(T, device=device)
        freqs[0] = 1.0
        pink_filter = 1.0 / torch.sqrt(freqs)
        white = torch.randn(B, T, device=device)
        pink = torch.fft.irfft(torch.fft.rfft(white) * pink_filter, n=T)
        rms_sig = torch.sqrt(torch.mean(wavs ** 2, dim=1, keepdim=True) + 1e-9)
        rms_noise = torch.sqrt(torch.mean(pink ** 2, dim=1, keepdim=True) + 1e-9)
        snr_db = torch.empty(B, 1, device=device).uniform_(
            cfg['pink_noise_snr_min'], cfg['pink_noise_snr_max']
        )
        scale = rms_sig / (rms_noise * 10.0 ** (snr_db / 20.0))
        wavs = torch.clamp(wavs + pink * scale, -1.0, 1.0)

    # 4. Mixup — blend BOTH waveforms AND labels (not max)
    # Using max() creates noisy multi-label supervision.
    # Proper mixup blends labels proportionally for smoother gradients.
    if random.random() < cfg['mixup_prob']:
        indices = torch.randperm(B, device=device)
        lam = torch.empty(B, 1, device=device).uniform_(0.3, 0.7)
        wavs = torch.clamp(lam * wavs + (1.0 - lam) * wavs[indices], -1.0, 1.0)
        targets = lam * targets + (1.0 - lam) * targets[indices]
        secondary = torch.max(secondary, secondary[indices])  # keep max for secondary (masking only)

    return wavs, targets, secondary

## 6 - Training & Validation Loops

In [9]:
def build_optimizer(model, cfg):
    """AdamW with layer-wise learning rate decay."""
    return AdamW(
        [
            {'params': model.frontend.parameters(), 'lr': cfg['lr']},
            {'params': model.head.parameters(), 'lr': cfg['lr']},
            {'params': model.backbone.parameters(),
             'lr': cfg['lr'] * cfg['backbone_lr_mult']},
        ],
        weight_decay=cfg['weight_decay'],
    )


def train_one_epoch(model, loader, optimizer, scheduler, criterion,
                    scaler, device, epoch, cfg):
    model.train()
    running_loss = torch.tensor(0.0, device=device)

    for wav, p_tgt, s_tgt in tqdm(loader, desc=f'Epoch {epoch:02d}', leave=False):
        wav = wav.to(device, non_blocking=True)
        p_tgt = p_tgt.to(device, non_blocking=True)
        s_tgt = s_tgt.to(device, non_blocking=True)

        wav, p_tgt, s_tgt = batch_augment(wav, p_tgt, s_tgt, cfg)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            logits = model(wav)
            loss = criterion(logits, p_tgt, s_tgt)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.detach()

    return (running_loss / len(loader)).item()


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    all_logits, all_targets = [], []
    running_loss = torch.tensor(0.0, device=device)

    for wav, p_tgt, s_tgt in tqdm(loader, desc='[val]', leave=False):
        wav = wav.to(device, non_blocking=True)
        p_tgt = p_tgt.to(device, non_blocking=True)
        s_tgt = s_tgt.to(device, non_blocking=True)

        with torch.amp.autocast('cuda'):
            logits = model(wav)
            running_loss += criterion(logits, p_tgt, s_tgt).detach()

        all_logits.append(logits.detach())
        all_targets.append(p_tgt.detach())

    logits_np = torch.cat(all_logits).float().cpu().numpy()
    targets_np = torch.cat(all_targets).float().cpu().numpy()
    probs = 1.0 / (1.0 + np.exp(-logits_np))

    try:
        per_class = roc_auc_score(targets_np, probs, average=None)
        auc = float(np.nanmean(per_class))
    except ValueError:
        auc = 0.0

    return (running_loss / len(loader)).item(), auc

## 7 - Model Soup

Average weights from multiple checkpoints (different epochs + folds).
Achieves ensemble-like quality at **single-model** inference cost.
Critical for the 90-minute CPU inference budget.

In [10]:
def model_soup(checkpoints: List[str]) -> dict:
    """Average weights from multiple checkpoints."""
    if not checkpoints:
        raise ValueError('No checkpoints for model soup')

    avg_state = None
    for ckpt_path in checkpoints:
        state = torch.load(ckpt_path, map_location='cpu', weights_only=True)
        if avg_state is None:
            avg_state = {k: v.float() for k, v in state.items()}
        else:
            for k in avg_state:
                avg_state[k] += state[k].float()

    for k in avg_state:
        avg_state[k] /= len(checkpoints)

    return avg_state

In [11]:
import os

# List all inputs
for d in os.listdir('/kaggle/input/'):
    print(f"  /kaggle/input/{d}/")
    try:
        for f in os.listdir(f'/kaggle/input/{d}/'):
            print(f"    {f}")
    except:
        pass

  /kaggle/input/datasets/
    shishiradhikari11
  /kaggle/input/competitions/
    birdclef-2026


In [12]:
import os
for root, dirs, files in os.walk('/kaggle/input/datasets/shishiradhikari11/'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/shishiradhikari11/birdclef-2026-checkpoints/eca_nfnet_l0_fold1_ep14.pt
/kaggle/input/datasets/shishiradhikari11/birdclef-2026-checkpoints/tf_efficientnetv2_b3_fold0_ep14.pt
/kaggle/input/datasets/shishiradhikari11/birdclef-2026-checkpoints/eca_nfnet_l0_fold1_ep16.pt
/kaggle/input/datasets/shishiradhikari11/birdclef-2026-checkpoints/eca_nfnet_l0_fold0_ep14.pt
/kaggle/input/datasets/shishiradhikari11/birdclef-2026-checkpoints/tf_efficientnet_b0_ns_fold1_ep20.pt
/kaggle/input/datasets/shishiradhikari11/birdclef-2026-checkpoints/eca_nfnet_l0_fold1_best.pt
/kaggle/input/datasets/shishiradhikari11/birdclef-2026-checkpoints/tf_efficientnet_b0_ns_fold2_ep20.pt
/kaggle/input/datasets/shishiradhikari11/birdclef-2026-checkpoints/tf_efficientnetv2_b3_fold1_ep16.pt
/kaggle/input/datasets/shishiradhikari11/birdclef-2026-checkpoints/tf_efficientnetv2_b3_fold1_ep18.pt
/kaggle/input/datasets/shishiradhikari11/birdclef-2026-checkpoints/tf_efficientnet_b0_ns_fold1_ep18.pt
/kaggle/i

## 8 - DDP Worker (Written to disk)

PyTorch's `mp.start_processes` with `start_method='spawn'` requires the worker function to be importable from a file.

In [13]:
%%writefile /kaggle/working/worker.py

import ast
import gc
import json
import math
import os
import random
import time
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import librosa
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torch.utils.data.distributed import DistributedSampler

import timm

warnings.filterwarnings('ignore')


# =========================================================================
# MULTI-SCALE MEL FRONTEND
# =========================================================================
class MultiScaleMelFrontend(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        sr = cfg['sample_rate']
        n_mels = cfg['n_mels']
        fmin, fmax = cfg['fmin'], cfg['fmax']
        self.n_fft_lo = cfg['n_fft_lo']
        self.hop_lo = cfg['hop_length_lo']
        self.n_fft_hi = cfg['n_fft_hi']
        self.hop_hi = cfg['hop_length_hi']
        self.n_mels = n_mels
        self.eps = 1e-6

        mel_lo = librosa.filters.mel(
            sr=sr, n_fft=self.n_fft_lo, n_mels=n_mels,
            fmin=fmin, fmax=fmax, norm='slaney', htk=False,
        ).astype(np.float32)
        mel_hi = librosa.filters.mel(
            sr=sr, n_fft=self.n_fft_hi, n_mels=n_mels,
            fmin=fmin, fmax=fmax, norm='slaney', htk=False,
        ).astype(np.float32)
        self.register_buffer('mel_fb_lo', torch.from_numpy(mel_lo))
        self.register_buffer('mel_fb_hi', torch.from_numpy(mel_hi))
        self.register_buffer('win_lo', torch.hann_window(self.n_fft_lo))
        self.register_buffer('win_hi', torch.hann_window(self.n_fft_hi))
        self.norm = nn.InstanceNorm2d(2, affine=False)

    def _mel_spec(self, wav, n_fft, hop, win, mel_fb):
        stft = torch.stft(
            wav, n_fft=n_fft, hop_length=hop,
            win_length=n_fft, window=win,
            return_complex=True, normalized=False,
        )
        power = stft.abs().pow(2)
        mel = torch.matmul(mel_fb, power)
        return torch.log(mel.clamp(min=self.eps))

    @torch.amp.autocast('cuda', enabled=False)
    def forward(self, wav):
        wav = wav.float()
        mel_lo = self._mel_spec(wav, self.n_fft_lo, self.hop_lo,
                                self.win_lo, self.mel_fb_lo)
        mel_hi = self._mel_spec(wav, self.n_fft_hi, self.hop_hi,
                                self.win_hi, self.mel_fb_hi)
        T_lo, T_hi = mel_lo.shape[-1], mel_hi.shape[-1]
        T_target = max(T_lo, T_hi)
        if T_lo < T_target:
            mel_lo = F.interpolate(
                mel_lo.unsqueeze(1), size=(self.n_mels, T_target),
                mode='bilinear', align_corners=False,
            ).squeeze(1)
        if T_hi < T_target:
            mel_hi = F.interpolate(
                mel_hi.unsqueeze(1), size=(self.n_mels, T_target),
                mode='bilinear', align_corners=False,
            ).squeeze(1)
        x = torch.stack([mel_lo, mel_hi], dim=1)
        x = self.norm(x)
        return x


# =========================================================================
# DATASET
# =========================================================================
class BirdDataset(Dataset):
    def __init__(self, meta_df, npy_path, cfg, is_train=True):
        self.is_train = is_train
        self.npy_path = npy_path
        self.mmap_indices = meta_df['mmap_index'].values.astype(np.int64)
        self.label_ids = meta_df['label_id'].values.astype(np.int64)
        self._waveforms = None
        n = len(self.mmap_indices)
        num_classes = cfg['num_classes']
        self.primary_targets = torch.zeros(n, num_classes, dtype=torch.float32)
        self.primary_targets[torch.arange(n), torch.from_numpy(self.label_ids)] = 1.0
        self.secondary_targets = torch.zeros(n, num_classes, dtype=torch.float32)
        lbl2id = cfg.get('label_map', {})
        if 'secondary_labels' in meta_df.columns:
            for i, sec_str in enumerate(meta_df['secondary_labels'].fillna('[]')):
                try:
                    sec_list = ast.literal_eval(sec_str)
                    for species in sec_list:
                        key = int(species) if isinstance(species, (int, float)) else species
                        if key in lbl2id:
                            self.secondary_targets[i, lbl2id[key]] = 1.0
                except Exception:
                    pass

    def _init_arrays(self):
        if self._waveforms is None:
            self._waveforms = np.load(self.npy_path, mmap_mode='r')

    def __len__(self):
        return len(self.mmap_indices)

    def __getitem__(self, index):
        self._init_arrays()
        mmap_idx = int(self.mmap_indices[index])
        wav = torch.from_numpy(self._waveforms[mmap_idx].copy())
        return wav, self.primary_targets[index], self.secondary_targets[index]


def worker_init_fn(worker_id):
    try:
        import psutil
        psutil.Process().cpu_affinity([worker_id % os.cpu_count()])
    except Exception:
        pass
    random.seed(torch.initial_seed() % (2 ** 32))


def build_class_weights(df, num_classes):
    counts = np.zeros(num_classes, dtype=np.float64)
    for lid in df['label_id'].values:
        counts[lid] += 1
    freq = counts / counts.sum()
    weights = 1.0 / np.sqrt(freq + 1e-8)
    weights /= weights.sum()
    sample_weights = torch.zeros(len(df), dtype=torch.float64)
    for i, lid in enumerate(df['label_id'].values):
        sample_weights[i] = weights[lid]
    return sample_weights


# =========================================================================
# MODEL
# =========================================================================
class SEDHead(nn.Module):
    def __init__(self, in_features, num_classes):
        super().__init__()
        self.fc = nn.Linear(in_features, num_classes)
        self.attention = nn.Sequential(
            nn.Linear(in_features, 128), nn.Tanh(),
            nn.Linear(128, num_classes),
        )
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = x.clamp(min=1e-6).pow(3.0).mean(dim=2).pow(1.0 / 3.0)
        x = x.transpose(1, 2)
        x = self.dropout(x)  # Dropout BEFORE classification, inside the head
        framewise = self.fc(x)
        att_weights = torch.softmax(self.attention(x), dim=1)
        return torch.sum(framewise * att_weights, dim=1)


class BirdModel(nn.Module):
    def __init__(self, cfg, backbone_name, is_train=True):
        super().__init__()
        self.is_train = is_train
        self.frontend = MultiScaleMelFrontend(cfg)
        self.spec_aug = self._build_spec_augment(cfg)
        self.backbone = timm.create_model(
            backbone_name, pretrained=True,
            in_chans=2, num_classes=0, global_pool='',
        )
        self.head = SEDHead(self.backbone.num_features, cfg['num_classes'])

    def _build_spec_augment(self, cfg):
        import torchaudio.transforms as T
        layers = []
        for _ in range(cfg['n_freq_masks']):
            layers.append(T.FrequencyMasking(freq_mask_param=cfg['freq_mask_param']))
        for _ in range(cfg['n_time_masks']):
            layers.append(T.TimeMasking(time_mask_param=cfg['time_mask_param']))
        return nn.Sequential(*layers)

    def forward(self, wav):
        spec = self.frontend(wav)
        if self.is_train and self.training:
            # Apply SAME mask to both channels (consistent masking)
            B, C, M, T = spec.shape
            # Generate mask on channel 0, apply to both
            ch0 = spec[:, 0:1, :, :]  # (B, 1, M, T)
            ch0_masked = self.spec_aug(ch0)
            # Compute the mask: where did values change?
            mask = (ch0_masked != ch0).float()
            # Apply same mask to all channels
            spec = spec * (1.0 - mask.expand_as(spec))
        features = self.backbone(spec)
        return self.head(features)


# =========================================================================
# LOSS
# =========================================================================
class FocalBCELoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, primary_targets, secondary_targets):
        combined = torch.clamp(primary_targets + secondary_targets, 0.0, 1.0)
        bce = F.binary_cross_entropy_with_logits(logits, combined, reduction='none')
        probs = torch.sigmoid(logits)
        p_t = probs * combined + (1 - probs) * (1 - combined)
        focal_weight = (1 - p_t).pow(self.gamma)
        alpha_t = self.alpha * combined + (1 - self.alpha) * (1 - combined)
        focal_loss = alpha_t * focal_weight * bce
        mask = torch.ones_like(focal_loss)
        mask[secondary_targets == 1.0] = 0.0
        mask[primary_targets == 1.0] = 1.0
        return (focal_loss * mask).mean()


# =========================================================================
# AUGMENTATION
# =========================================================================
def batch_augment(wavs, targets, secondary, cfg):
    B, T = wavs.shape
    device = wavs.device

    if random.random() < cfg['time_shift_prob']:
        shifts = torch.randint(0, T, (B, 1), device=device)
        idx = (torch.arange(T, device=device).unsqueeze(0) - shifts) % T
        wavs = torch.gather(wavs, 1, idx)

    gain_mask = (torch.rand(B, 1, device=device) < cfg['gain_prob']).float()
    gain_db = torch.empty(B, 1, device=device).uniform_(cfg['gain_db_min'], cfg['gain_db_max'])
    linear_gain = 1.0 + (10.0 ** (gain_db / 20.0) - 1.0) * gain_mask
    wavs = torch.clamp(wavs * linear_gain, -1.0, 1.0)

    if random.random() < cfg['pink_noise_prob']:
        freqs = torch.fft.rfftfreq(T, device=device)
        freqs[0] = 1.0
        pink_filter = 1.0 / torch.sqrt(freqs)
        white = torch.randn(B, T, device=device)
        pink = torch.fft.irfft(torch.fft.rfft(white) * pink_filter, n=T)
        rms_sig = torch.sqrt(torch.mean(wavs ** 2, dim=1, keepdim=True) + 1e-9)
        rms_noise = torch.sqrt(torch.mean(pink ** 2, dim=1, keepdim=True) + 1e-9)
        snr_db = torch.empty(B, 1, device=device).uniform_(
            cfg['pink_noise_snr_min'], cfg['pink_noise_snr_max'])
        scale = rms_sig / (rms_noise * 10.0 ** (snr_db / 20.0))
        wavs = torch.clamp(wavs + pink * scale, -1.0, 1.0)

    # Proper mixup: blend labels proportionally (not max)
    if random.random() < cfg['mixup_prob']:
        indices = torch.randperm(B, device=device)
        lam = torch.empty(B, 1, device=device).uniform_(0.3, 0.7)
        wavs = torch.clamp(lam * wavs + (1.0 - lam) * wavs[indices], -1.0, 1.0)
        targets = lam * targets + (1.0 - lam) * targets[indices]
        secondary = torch.max(secondary, secondary[indices])

    return wavs, targets, secondary


# =========================================================================
# TRAINING LOOPS
# =========================================================================
def build_optimizer(model, cfg):
    return AdamW(
        [
            {'params': model.frontend.parameters(), 'lr': cfg['lr']},
            {'params': model.head.parameters(), 'lr': cfg['lr']},
            {'params': model.backbone.parameters(),
             'lr': cfg['lr'] * cfg['backbone_lr_mult']},
        ],
        weight_decay=cfg['weight_decay'],
    )


def train_one_epoch(model, loader, optimizer, scheduler, criterion,
                    scaler, device, epoch, cfg):
    model.train()
    running_loss = torch.tensor(0.0, device=device)
    for wav, p_tgt, s_tgt in tqdm(loader, desc=f'Epoch {epoch:02d}', leave=False):
        wav = wav.to(device, non_blocking=True)
        p_tgt = p_tgt.to(device, non_blocking=True)
        s_tgt = s_tgt.to(device, non_blocking=True)
        wav, p_tgt, s_tgt = batch_augment(wav, p_tgt, s_tgt, cfg)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            logits = model(wav)
            loss = criterion(logits, p_tgt, s_tgt)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running_loss += loss.detach()
    return (running_loss / len(loader)).item()


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    all_logits, all_targets = [], []
    running_loss = torch.tensor(0.0, device=device)
    for wav, p_tgt, s_tgt in tqdm(loader, desc='[val]', leave=False):
        wav = wav.to(device, non_blocking=True)
        p_tgt = p_tgt.to(device, non_blocking=True)
        s_tgt = s_tgt.to(device, non_blocking=True)
        with torch.amp.autocast('cuda'):
            logits = model(wav)
            running_loss += criterion(logits, p_tgt, s_tgt).detach()
        all_logits.append(logits.detach())
        all_targets.append(p_tgt.detach())
    logits_np = torch.cat(all_logits).float().cpu().numpy()
    targets_np = torch.cat(all_targets).float().cpu().numpy()
    probs = 1.0 / (1.0 + np.exp(-logits_np))
    try:
        per_class = roc_auc_score(targets_np, probs, average=None)
        auc = float(np.nanmean(per_class))
    except ValueError:
        auc = 0.0
    return (running_loss / len(loader)).item(), auc


# =========================================================================
# DDP SETUP
# =========================================================================
def ddp_setup(rank, world_size):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    os.environ.setdefault('NCCL_P2P_DISABLE', '1')
    os.environ.setdefault('NCCL_IB_DISABLE', '1')
    dist.init_process_group(backend='nccl', rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)


def ddp_cleanup():
    dist.destroy_process_group()


# =========================================================================
# TRAIN WORKER — Fixed DDP + WeightedSampler conflict
# =========================================================================
def train_worker(rank, world_size, cfg, meta_df, npy_path, backbone_name, fold):
    ddp_setup(rank, world_size)
    torch.cuda.manual_seed_all(42 + fold)
    torch.backends.cudnn.benchmark = True
    device = torch.device(f'cuda:{rank}')

    skf = StratifiedKFold(n_splits=cfg['n_folds'], shuffle=True, random_state=42)
    splits = list(skf.split(meta_df, meta_df['label_id']))
    train_idx, val_idx = splits[fold]
    df_train = meta_df.iloc[train_idx].copy()
    df_val = meta_df.iloc[val_idx].copy()

    if rank == 0:
        print(f'\n[{backbone_name} | Fold {fold}] '
              f'Train: {len(df_train):,} | Val: {len(df_val):,}')

    train_ds = BirdDataset(df_train, npy_path, cfg, is_train=True)
    val_ds = BirdDataset(df_val, npy_path, cfg, is_train=False)

    # FIX: DDP-compatible class-balanced sampling
    # DistributedSampler partitions data across GPUs
    # WeightedRandomSampler then balances classes within each GPU's partition
    # Without DistributedSampler, both GPUs see the same samples = wasted compute
    train_sampler = DistributedSampler(
        train_ds, num_replicas=world_size, rank=rank,
        shuffle=True, drop_last=True,
    )
    val_sampler = DistributedSampler(
        val_ds, num_replicas=world_size, rank=rank,
        shuffle=False, drop_last=False,
    )

    bs = cfg['batch_size'] // world_size
    train_loader = DataLoader(
        train_ds, batch_size=bs, sampler=train_sampler,
        num_workers=cfg['num_workers'], pin_memory=True,
        persistent_workers=True, drop_last=True,
        worker_init_fn=worker_init_fn,
    )
    val_loader = DataLoader(
        val_ds, batch_size=bs * 2, sampler=val_sampler,
        num_workers=cfg['num_workers'], pin_memory=True,
        persistent_workers=True, worker_init_fn=worker_init_fn,
    )

    model = BirdModel(cfg, backbone_name, is_train=True).to(device)
    model = DDP(model, device_ids=[rank])

    optimizer = build_optimizer(model.module, cfg)

    # FIX: OneCycleLR max_lr must match optimizer param group LRs
    # Don't pass separate max_lr — let it use optimizer's LRs as max
    scheduler = OneCycleLR(
        optimizer,
        max_lr=[cfg['lr'], cfg['lr'], cfg['lr'] * cfg['backbone_lr_mult']],
        epochs=cfg['epochs'],
        steps_per_epoch=len(train_loader),
        pct_start=0.1,
    )
    criterion = FocalBCELoss(
        gamma=cfg['focal_gamma'], alpha=cfg['focal_alpha']).to(device)
    scaler = torch.amp.GradScaler('cuda')

    best_auc = 0.0
    run_name = f"{backbone_name.replace('/', '_')}_fold{fold}"

    for epoch in range(1, cfg['epochs'] + 1):
        train_sampler.set_epoch(epoch)  # CRITICAL for DDP reshuffling

        train_loss = train_one_epoch(
            model, train_loader, optimizer, scheduler, criterion,
            scaler, device, epoch, cfg,
        )
        val_loss, val_auc = validate(model, val_loader, criterion, device)

        if rank == 0:
            print(f'  [{run_name}] Ep {epoch:02d} | '
                  f'TrLoss: {train_loss:.4f} | VlLoss: {val_loss:.4f} | '
                  f'AUC: {val_auc:.4f}')
            if val_auc > best_auc:
                best_auc = val_auc
                torch.save(
                    model.module.state_dict(),
                    cfg['output_dir'] / f'{run_name}_best.pt',
                )
            if epoch in cfg['soup_epochs']:
                torch.save(
                    model.module.state_dict(),
                    cfg['output_dir'] / f'{run_name}_ep{epoch}.pt',
                )

    if rank == 0:
        print(f'  [{run_name}] Best AUC: {best_auc:.4f}')

    ddp_cleanup()

Writing /kaggle/working/worker.py


## 9 - Load Data & Launch Training

In [14]:
import sys
sys.path.insert(0, '/kaggle/working')

# Load preprocessed data
prep_dir = CFG['preprocessed_dir']
meta_df = pd.read_csv(prep_dir / 'metadata_clean.csv')
npy_path = str(prep_dir / 'waveforms.npy')

with open(prep_dir / 'label_map.json') as f:
    label_map_str = json.load(f)

# Reconstruct label_map with proper key types
label_map = {}
for k, v in label_map_str.items():
    try:
        label_map[int(k)] = v
    except ValueError:
        label_map[k] = v

CFG['num_classes'] = len(label_map)
CFG['label_map'] = label_map

# Load target columns for submission alignment
with open(prep_dir / 'target_columns.json') as f:
    target_columns = json.load(f)

print(f'[DATA] {len(meta_df):,} samples | {CFG["num_classes"]} classes')
print(f'[DATA] Target columns: {len(target_columns)}')

[DATA] 34,604 samples | 234 classes
[DATA] Target columns: 234


In [15]:
from worker import train_worker

for backbone_name in CFG['backbones']:
    for fold in CFG['train_folds']:
        print(f"\n{'='*60}")
        print(f'TRAINING: {backbone_name} | Fold {fold}')
        print(f"{'='*60}")

        mp.start_processes(
            train_worker,
            args=(N_GPUS, CFG, meta_df, npy_path, backbone_name, fold),
            nprocs=N_GPUS,
            join=True,
            start_method='spawn',
        )
        gc.collect()
        torch.cuda.empty_cache()


TRAINING: tf_efficientnet_b0_ns | Fold 0

TRAINING: tf_efficientnet_b0_ns | Fold 1

TRAINING: tf_efficientnet_b0_ns | Fold 2

TRAINING: eca_nfnet_l0 | Fold 0

TRAINING: eca_nfnet_l0 | Fold 1

TRAINING: eca_nfnet_l0 | Fold 2

TRAINING: tf_efficientnetv2_b3 | Fold 0

TRAINING: tf_efficientnetv2_b3 | Fold 1

TRAINING: tf_efficientnetv2_b3 | Fold 2


In [16]:
import os
for f in sorted(os.listdir('/kaggle/working/')):
    if f.endswith('.pt'):
        size_mb = os.path.getsize(f'/kaggle/working/{f}') / 1e6
        print(f"  {f:45s} {size_mb:.1f} MB")
    elif f.endswith('.npy') or f.endswith('.csv') or f.endswith('.json'):
        print(f"  {f}")

## 10 - Build Model Soups

In [17]:
print('Building Model Soups...')
print('=' * 60)

for backbone_name in CFG['backbones']:
    run_prefix = backbone_name.replace('/', '_')
    ckpts = []
    for fold in CFG['train_folds']:
        for ep in CFG['soup_epochs']:
            path = CFG['output_dir'] / f'{run_prefix}_fold{fold}_ep{ep}.pt'
            if path.exists():
                ckpts.append(str(path))
        best_path = CFG['output_dir'] / f'{run_prefix}_fold{fold}_best.pt'
        if best_path.exists():
            ckpts.append(str(best_path))

    if ckpts:
        print(f'  [{backbone_name}] Souping {len(ckpts)} checkpoints...')
        soup_state = model_soup(ckpts)
        torch.save(soup_state, CFG['output_dir'] / f'{run_prefix}_soup.pt')
        print(f'  [{backbone_name}] Soup saved.')
    else:
        print(f'  [{backbone_name}] No checkpoints found!')

print(f'\nDone! Model soups saved to {CFG["output_dir"]}')

Building Model Soups...
  [tf_efficientnet_b0_ns] No checkpoints found!
  [eca_nfnet_l0] No checkpoints found!
  [tf_efficientnetv2_b3] No checkpoints found!

Done! Model soups saved to /kaggle/working


In [18]:
# Copy config files to output (needed by inference notebook)
import shutil
shutil.copy(prep_dir / 'label_map.json', CFG['output_dir'] / 'label_map.json')
shutil.copy(prep_dir / 'target_columns.json', CFG['output_dir'] / 'target_columns.json')

!ls -lh /kaggle/working/*.pt /kaggle/working/*.json

ls: cannot access '/kaggle/working/*.pt': No such file or directory
-rw-r--r-- 1 root root 4.0K Mar 19 23:59  /kaggle/working/label_map.json
-rw-r--r-- 1 root root 2.5K Mar 19 23:59  /kaggle/working/target_columns.json
